# 09.3 - Embeddings

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Embeddings are dense, fixed-size vector representations of tokens (or text). An embedding layer converts each token ID into a continuous vector the transformer can process. Semantically similar tokens land near each other in vector space.

## 2. Why Does This Matter?

Embeddings are how models represent meaning numerically. Without them the model would only see discrete integers with no relationship. Semantic similarity ("king" close to "queen", far from "table") is what makes retrieval, search, and reasoning possible.

## 3. Prerequisites

- Unit 09.2 (tokenization)
- Basic vector / dot-product intuition

## 4. Learning Objectives

- Explain the embedding matrix and lookup table
- Compute cosine similarity and interpret it
- Distinguish dense vs sparse / static vs contextual embeddings
- Inspect a real embedding tensor shape

## 5. Mental Model

An embedding is like a GPS coordinate for meaning: nearby points = similar meaning. A model keeps a matrix of shape `[vocab_size x embedding_dim]`; token ID *i* is just row *i* of that matrix. During training the matrix is learned like any other parameter.

```text
token ID  ->  row lookup in [vocab_size x dim] matrix  ->  dense vector
    42                        E                               [0.2, -0.1, 0.7, ...]
```


## 6. Setup

We use a tiny GPT-2-style model's embedding table - no weights are downloaded.


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import numpy as np
import torch.nn.functional as F
torch.manual_seed(0)
np.random.seed(0)
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(n_layer=1, n_head=1, n_embd=16, vocab_size=50, n_positions=32)
model = GPT2LMHeadModel(config)
model.eval()
print("Embedding matrix shape:", tuple(model.transformer.wte.weight.shape))
print("(vocab_size=50, embedding_dim=16)")


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49), got 50256. This may result in unexpected behavior.


[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49), got 50256. This may result in unexpected behavior.


Embedding matrix shape: (50, 16)
(vocab_size=50, embedding_dim=16)


## 7. Embedding as a Lookup Table

Token ID to vector is a row lookup. `nn.Embedding` does exactly this.


In [2]:
emb = model.transformer.wte.weight  # [50, 16]
token_id = torch.tensor([5, 17, 42])  # arbitrary token IDs
vectors = F.embedding(token_id, emb)
print("Input token IDs:", token_id.tolist())
print("Looked-up vectors shape:", tuple(vectors.shape))
print("Vector for token 5:", vectors[0].tolist())


Input token IDs: [5, 17, 42]
Looked-up vectors shape: (3, 16)
Vector for token 5: [-0.01848512329161167, -0.04619567468762398, 0.025455383583903313, 0.010968533344566822, -0.02957053855061531, -0.06945892423391342, -0.02504650317132473, 0.00537178386002779, -0.012891234830021858, 0.03399985656142235, -0.030970433726906776, 0.001358207082375884, -0.03068825975060463, 0.002560530323535204, -0.006323843263089657, -0.008018050342798233]


## 8. Cosine Similarity

Cosine similarity measures the angle between two vectors (ignoring magnitude). Range [-1, 1]: 1 = same direction, 0 = orthogonal, -1 = opposite.


In [3]:
def cos_sim(a, b):
    a, b = a / a.norm(), b / b.norm()
    return float((a * b).sum())

# Compare token 5 against every other token
vec5 = emb[5]
scores = [cos_sim(vec5, emb[i]) for i in range(emb.shape[0])]
top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
print("Tokens most similar to token 5:")
for tid in top:
    print(f"  token {tid:2d}: cos_sim = {scores[tid]:+.3f}")

# A very different token
print("\nCosine similarity is in [-1,1] and proxies semantic closeness.")


Tokens most similar to token 5:
  token  5: cos_sim = +1.000
  token 45: cos_sim = +0.554
  token 17: cos_sim = +0.502
  token 42: cos_sim = +0.377
  token 47: cos_sim = +0.361

Cosine similarity is in [-1,1] and proxies semantic closeness.


C:\Users\PC\AppData\Local\Temp\ipykernel_13224\3809352318.py:3: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  return float((a * b).sum())


## 9. Dense vs Sparse Representations

One-hot encoding is sparse (one 1, rest 0) and has no notion of similarity - 'cat' and 'dog' are orthogonal. Dense embeddings are learned so similar words share direction.


In [4]:
def one_hot(token_id, size=50):
    v = np.zeros(size)
    v[token_id] = 1.0
    return v

oh_a, oh_b, oh_c = one_hot(5), one_hot(6), one_hot(7)
print("One-hot similarity (5 vs 6):", float(np.dot(oh_a, oh_b)))
print("One-hot similarity (5 vs 7):", float(np.dot(oh_a, oh_c)))
print("All zero -> one-hot cannot express '5 is closer to 6 than to 7'.")

# Dense embeddings do
print("Dense cos_sim (5 vs 6):", f"{cos_sim(emb[5], emb[6]):+.3f}")
print("Dense cos_sim (5 vs 7):", f"{cos_sim(emb[5], emb[7]):+.3f}")


One-hot similarity (5 vs 6): 0.0
One-hot similarity (5 vs 7): 0.0
All zero -> one-hot cannot express '5 is closer to 6 than to 7'.
Dense cos_sim (5 vs 6): -0.063
Dense cos_sim (5 vs 7): +0.157


## 10. Averaging Embeddings to Represent a Sequence

A simple way to embed a whole text is to average its token embeddings. (Modern models instead use contextual embeddings - see section 12.)


In [5]:
def embed_sequence(ids):
    vecs = F.embedding(torch.tensor(ids), emb)
    return vecs.mean(dim=0)

seq_a = embed_sequence([3, 5, 10])   # 'group one'
seq_b = embed_sequence([4, 5, 11])   # 'group two'
seq_c = embed_sequence([20, 21, 22]) # unrelated
print("Seq A vs Seq B (both contain token 5):", f"{cos_sim(seq_a, seq_b):+.3f}")
print("Seq A vs Seq C (unrelated):", f"{cos_sim(seq_a, seq_c):+.3f}")
print()
print("Shared tokens pull sequence embeddings closer (toy example).")
print("Note: averaging is a toy; real models use contextual encoders.")


Seq A vs Seq B (both contain token 5): +0.590
Seq A vs Seq C (unrelated): -0.195

Shared tokens pull sequence embeddings closer (toy example).
Note: averaging is a toy; real models use contextual encoders.


## 11. Failure Case: Mixing Models

Embedding spaces are **not compatible across models**. Two models embed the same idea into different, unrelated coordinate frames, so comparing vectors from different models is meaningless.


In [6]:
config2 = GPT2Config(n_layer=1, n_head=1, n_embd=16, vocab_size=50, n_positions=32)
model2 = GPT2LMHeadModel(config2)
emb2 = model2.transformer.wte.weight

# Same token id, two different random models -> totally unrelated vectors
print("Model A vector for token 5:", emb[5].tolist()[:4], "...")
print("Model B vector for token 5:", emb2[5].tolist()[:4], "...")
print("cos_sim across models:", f"{cos_sim(emb[5], emb2[5]):+.3f}")
print("\nAlways embed AND compare within the same model/tokenizer.")


Model A vector for token 5: [-0.01848512329161167, -0.04619567468762398, 0.025455383583903313, 0.010968533344566822] ...
Model B vector for token 5: [-0.005424216389656067, -0.007028177380561829, 0.04466031491756439, 0.0031720001716166735] ...
cos_sim across models: -0.066

Always embed AND compare within the same model/tokenizer.


## 12. Static vs Contextual Embeddings

A raw embedding table is **static**: 'bank' always maps to the same vector. Contextual embeddings (from passing tokens through transformer layers) give *different* vectors for the same word in different sentences - 'bank' in 'river bank' vs 'money bank'.


In [7]:
with torch.no_grad():
    x = torch.tensor([[2, 8, 8, 3]])   # note: token 8 appears twice
    h = model.transformer(x).last_hidden_state  # contextual after 1 layer
ctx_a = h[0, 1]   # first occurrence of token 8
ctx_b = h[0, 2]   # second occurrence (same token id, different positions/context)
print("Contextual vector (pos 1):", ctx_a.tolist()[:4], "...")
print("Contextual vector (pos 2):", ctx_b.tolist()[:4], "...")
print("Same token id at two positions - differ? ", not torch.allclose(ctx_a, ctx_b, atol=1e-4))
print("\nTransformers produce position-aware contextual embeddings.")


Contextual vector (pos 1): [-0.8847947716712952, -2.069312572479248, 0.11888744682073593, -0.2789669334888458] ...
Contextual vector (pos 2): [-0.5699384212493896, 0.006434130482375622, 2.360940933227539, 0.2076374888420105] ...
Same token id at two positions - differ?  True

Transformers produce position-aware contextual embeddings.


## 13. Debugging & Best Practices

| Symptom | Cause | Fix |
|---|---|---|
| Random similarity scores | Wrong model/tokenizer | Match tokenizer & model |
| Wrong tensor shape | Padding/batch issue | Print shape |
| Similarity always ~1 | Identical/normalized vectors | Inspect raw vectors |
| OOM with big matrices | vocab_size x dim large | Reduce batch / smaller model |

- Always use the same tokenizer + model to embed and compare.
- Normalize vectors before computing cosine similarity.
- Store embeddings as float16/32 and index for fast search.

## 14. Common Mistakes

- Confusing token IDs with embeddings.
- Assuming embeddings capture truth or full meaning.
- Mixing embedding spaces across models.
- Ignoring embedding dimension mismatch.

## 15. When NOT to Use Embeddings

- When you need exact keyword matching only -> sparse/lexical search.
- When you have only classification with few features -> simpler encodings suffice.

## 16. Challenge

Normalize all token vectors, then verify that with L2 normalization cosine similarity equals the plain dot product.


In [8]:
def normalized_dot(a, b):
    a = a / a.norm()
    b = b / b.norm()
    return float((a * b).sum())

all_equal = all(
    np.isclose(cos_sim(emb[i], emb[j]), normalized_dot(emb[i], emb[j]), atol=1e-5)
    for i in range(10) for j in range(10)
)
print("cosine similarity == normalized dot product for 10x10 sample:", all_equal)
print("Confirmed: with unit-normalized vectors the two are identical.")


cosine similarity == normalized dot product for 10x10 sample: True
Confirmed: with unit-normalized vectors the two are identical.


## 17. Closed-Book Recall

1. What is the relationship between token IDs and embeddings?
2. Why are embeddings continuous vectors instead of one-hot?
3. What does cosine similarity measure?
4. Why are contextual embeddings different from static embeddings?

## 18. Teach-Back Questions

Explain to another person:

- What an embedding matrix stores, in your own words.
- Why you must not compare embeddings from two different models.

## 19. Summary

You inspected a real embedding matrix, implemented cosine similarity, contrasted dense vs sparse and static vs contextual embeddings, and demonstrated why model-mismatch breaks comparisons.

## 20. Further Experiment

- Project token embeddings into 2D with PCA / t-SNE and observe clusters.
- Study how contextual embeddings differ across transformer layers.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, transformers, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
